In [1]:
import gc
import sys
from statistics import median

import torch
from magi_attention.api import flex_flash_attn_func
from plotly import graph_objects as go
from safetensors.torch import load_file
from torch.nn.functional import scaled_dot_product_attention
from tqdm import tqdm

sys.path.append("/workspace-SR008.fs2/dmikhaylov/src/kandinsky-6/kandinsky/datasets")
from mask_bank import generate_random_sequence, make_sdpa_mask, qk_segments, reorder_sequence, create_mask_bank
sys.path.append("/home/jovyan/dmikhaylov/projects/k7_attn/omni_attention")
from src.omni_attention import omni_attention, BlockMask


Loading pre-compiled FFA_FA4 kernels from /home/jovyan/dmikhaylov/envs/dmikhaylov_cu131_magi/lib/python3.13/site-packages/magi_attention/lib/ffa_fa4_cache ...
	=> fwd: 0 kernels loaded
	=> bwd: 0 kernels loaded
	=> bwd_pre: 0 kernels loaded
	=> bwd_post: 0 kernels loaded
No pre-compiled FFA_FA4 kernels to load.


In [2]:
NUM_HEADS = 32
SEQ_LENS = [32000, 65535, 256000, 1024000]
HEAD_DIM = 128
NUM_REPEATS = 20

In [3]:
def rmse(x, y):
    return ((x.float() - y.float())**2).mean().sqrt().item()

def clean_mem(q, k, v):
    del q, k, v
    torch.cuda.empty_cache()

class BaseMeasurer:
    def __init__(self):
        self.start = torch.cuda.Event(enable_timing=True)
        self.end = torch.cuda.Event(enable_timing=True)
        self.attn_func = None
        self.key = "base"
        self.fwd_times = []
        self.bwd_times = []
        self.fwd_err = []
        self.bwd_err = []

    def fmt_qkvm(self, q, k, v, q_s, k_s, seg_types):
        return q, k, v

    def fmt_res(self, res):
        return res

    def fmt_grads(self, q, k, v):
        return q, k, v

    def attn_fwd(self, q, k, v):
        return None

    def measure(self, q, k, v, aux_vals, t_res=None, q_g=None, k_g=None, v_g=None):
        q, k, v = self.fmt_qkvm(q, k, v, aux_vals)
        self.start.record()
        res = self.attn_fwd(q, k, v)
        self.end.record()
        torch.cuda.synchronize()
        self.fwd_times.append(self.start.elapsed_time(self.end))

        with torch.no_grad():
            if t_res is not None:
                fres = self.fmt_res(res)
                self.fwd_err.append(rmse(fres, t_res))
                del fres

        loss = res.mean()
        self.start.record()
        loss.backward()
        self.end.record()
        torch.cuda.synchronize()
        self.bwd_times.append(self.start.elapsed_time(self.end))

        with torch.no_grad():
            if q_g is not None and k_g is not None and v_g is not None:
                fq, fk, fv = self.fmt_grads(q.grad, k.grad, v.grad)
                self.bwd_err.append(rmse(fq, q_g) + rmse(fk, k_g) + rmse(fv, v_g))
                del fq, fk, fv

        qg, kg, vg = q.grad.detach().clone(), k.grad.detach().clone(), v.grad.detach().clone()
        q.grad, k.grad, v.grad = None, None, None
        return res, qg, kg, vg

    def save_measures(self, fwd_times, bwd_times, fwd_err, bwd_err):
        fwd_times[self.key] = median(self.fwd_times) if self.fwd_times else None
        bwd_times[self.key] = median(self.bwd_times) if self.bwd_times else None
        fwd_err[self.key] = median(self.fwd_err) if self.fwd_err else None
        bwd_err[self.key] = median(self.bwd_err) if self.bwd_err else None

class SdpaMeasurer(BaseMeasurer):
    def __init__(self):
        super().__init__()
        self.key = 'sdpa'
        self.attn_func = scaled_dot_product_attention

    def fmt_qkvm(self, q, k, v, aux_vals):
        q_s, k_s, seg_types = aux_vals
        q = q.cuda().requires_grad_()
        k = k.cuda().requires_grad_()
        v = v.cuda().requires_grad_()
        slen = q.shape[2]
        self.mask = make_sdpa_mask(q_s, k_s, seg_types).to(q.device)[:slen, :slen]
        return q, k, v

    def attn_fwd(self, q, k, v):
        res = self.attn_func(q, k, v, self.mask)
        self.mask = None
        return res

class SdpaCompiledMeasurer(SdpaMeasurer):
    def __init__(self):
        super().__init__()
        self.key = 'sdpa_c'
        self.attn_func = torch.compile(scaled_dot_product_attention, dynamic=True)

class MagiMeasurer(BaseMeasurer):
    def __init__(self):
        super().__init__()
        self.key = 'magi'
        self.attn_func = flex_flash_attn_func

    def fmt_qkvm(self, q, k, v, aux_vals):
        q_s, k_s, seg_types = aux_vals
        q = q.cuda().transpose(1, 2).squeeze(0).contiguous().requires_grad_()
        k = k.cuda().transpose(1, 2).squeeze(0).contiguous().requires_grad_()
        v = v.cuda().transpose(1, 2).squeeze(0).contiguous().requires_grad_()
        self.q_s, self.k_s, self.seg_types = q_s.cuda(), k_s.cuda(), seg_types.cuda()
        return q, k, v

    def fmt_res(self, res):
        return res.transpose(0, 1).unsqueeze(0)

    def fmt_grads(self, q, k, v):
        return q.transpose(0, 1).unsqueeze(0), k.transpose(0, 1).unsqueeze(0), v.transpose(0, 1).unsqueeze(0)

    def attn_fwd(self, q, k, v):
        res , _ = self.attn_func(q, k, v, q_ranges=self.q_s, k_ranges=self.k_s, attn_type_map=self.seg_types)
        return res

class MagiRmMeasurer(MagiMeasurer):
    def __init__(self):
        super().__init__()
        self.key = 'magi_rm'
        self.attn_func = flex_flash_attn_func

    def attn_fwd(self, q, k, v):
        res , _ = self.attn_func(q, k, v, q_ranges=self.q_s, k_ranges=self.k_s, attn_type_map=self.seg_types,
                                 auto_range_merge=True)
        return res

class OmniMeasurer(BaseMeasurer):
    def __init__(self):
        super().__init__()
        self.key = 'omni_am'
        self.attn_func = omni_attention

    def fmt_qkvm(self, q, k, v, aux_vals):
        r_seq, mat = aux_vals
        block_mask, unique_blocks = create_mask_bank(r_seq, mat)
        mask_bank = torch.stack(unique_blocks)
        self.block_mask = BlockMask(block_mask.unsqueeze(0).numpy(), mask_bank.numpy(), m=mask_bank.shape[0]).to("cuda")
        q = q.cuda().transpose(1, 2).contiguous().requires_grad_()
        k = k.cuda().transpose(1, 2).contiguous().requires_grad_()
        v = v.cuda().transpose(1, 2).contiguous().requires_grad_()
        return q, k, v

    def fmt_res(self, res):
        return res.transpose(1, 2)

    def fmt_grads(self, q, k, v):
        return q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

    def attn_fwd(self, q, k, v):
        res = self.attn_func(q, k, v, self.block_mask)
        return res

    def measure(self, q, k, v, aux_vals, t_res=None, q_g=None, k_g=None, v_g=None):
        q, k, v = self.fmt_qkvm(q, k, v, aux_vals)

        res = self.attn_fwd(q, k, v)
        loss = res.mean()
        loss.backward()
        q.grad, k.grad, v.grad = None, None, None

        self.start.record()
        res = self.attn_fwd(q, k, v)
        self.end.record()
        torch.cuda.synchronize()
        self.fwd_times.append(self.start.elapsed_time(self.end))

        with torch.no_grad():
            if t_res is not None:
                fres = self.fmt_res(res)
                self.fwd_err.append(rmse(fres, t_res))
                del fres

        loss = res.mean()
        self.start.record()
        loss.backward()
        self.end.record()
        torch.cuda.synchronize()
        self.bwd_times.append(self.start.elapsed_time(self.end))

        with torch.no_grad():
            if q_g is not None and k_g is not None and v_g is not None:
                fq, fk, fv = self.fmt_grads(q.grad, k.grad, v.grad)
                self.bwd_err.append(rmse(fq, q_g) + rmse(fk, k_g) + rmse(fv, v_g))
                del fq, fk, fv

        qg, kg, vg = q.grad.detach().clone(), k.grad.detach().clone(), v.grad.detach().clone()
        q.grad, k.grad, v.grad = None, None, None
        return res, qg, kg, vg

In [4]:
BS = 64

fwd_times = {}
bwd_times = {}
fwd_err = {}
bwd_err = {}

for seq_len in SEQ_LENS:
    skey = f"{seq_len}"
    fwd_times[skey] = {}
    bwd_times[skey] = {}
    fwd_err[skey] = {}
    bwd_err[skey] = {}

    sdpa = SdpaMeasurer()
    sdpa_c = SdpaCompiledMeasurer()
    magi = MagiMeasurer()
    magi_rm = MagiRmMeasurer()
    omni = OmniMeasurer()
    for _ in tqdm(range(NUM_REPEATS)):
        seq, seq_types = generate_random_sequence(seq_len)
        r_seq, _, mat = reorder_sequence(seq, seq_types)
        q_s, k_s, seg_types = qk_segments(r_seq, mat)
        slen = sum([sum(s) for s in seq])

        q = torch.randn(1, NUM_HEADS, slen, HEAD_DIM, dtype=torch.bfloat16)
        k = torch.randn(1, NUM_HEADS, slen, HEAD_DIM, dtype=torch.bfloat16)
        v = torch.randn(1, NUM_HEADS, slen, HEAD_DIM, dtype=torch.bfloat16)

        if slen < 65536:
            res, q_grad, k_grad, v_grad = sdpa.measure(q, k, v, (q_s, k_s, seg_types))
            sdpa_c.measure(q, k, v, (q_s, k_s, seg_types), res, q_grad, k_grad, v_grad)
        else:
            res, q_grad, k_grad, v_grad = None, None, None, None
        magi.measure(q, k, v, (q_s, k_s, seg_types), res, q_grad, k_grad, v_grad)
        magi_rm.measure(q, k, v, (q_s, k_s, seg_types), res, q_grad, k_grad, v_grad)
        omni.measure(q, k, v, (r_seq, mat), res, q_grad, k_grad, v_grad)
        res, q_grad, k_grad, v_grad = None, None, None, None
        clean_mem(q, k, v)
    sdpa.save_measures(fwd_times[skey], bwd_times[skey], fwd_err[skey], bwd_err[skey])
    sdpa_c.save_measures(fwd_times[skey], bwd_times[skey], fwd_err[skey], bwd_err[skey])
    magi.save_measures(fwd_times[skey], bwd_times[skey], fwd_err[skey], bwd_err[skey])
    magi_rm.save_measures(fwd_times[skey], bwd_times[skey], fwd_err[skey], bwd_err[skey])
    omni.save_measures(fwd_times[skey], bwd_times[skey], fwd_err[skey], bwd_err[skey])
    del sdpa, sdpa_c, magi, magi_rm, omni
    gc.collect()
    torch.cuda.empty_cache()
    torch._dynamo.reset()


  0%|          | 0/20 [04:37<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 15.62 GiB. GPU 0 has a total capacity of 79.19 GiB of which 1.16 GiB is free. Including non-PyTorch memory, this process has 78.02 GiB memory in use. Of the allocated memory 56.84 GiB is allocated by PyTorch, and 15.38 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [6]:
fig = go.Figure()

y = [fwd_times[k]["sdpa"] for k in fwd_times if "sdpa" in fwd_times[k]]
yt = [f"{yy:0.2f}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(fwd_times.keys()), y=y, text=yt, name="sdpa", showlegend=True))

y = [fwd_times[k]["sdpa_c"] for k in fwd_times if "sdpa_c" in fwd_times[k]]
yt = [f"{yy:0.2f}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(fwd_times.keys()), y=y, text=yt, name="sdpa_c", showlegend=True))

y = [fwd_times[k]["magi"] for k in fwd_times if "magi" in fwd_times[k]]
yt = [f"{yy:0.2f}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(fwd_times.keys()), y=y, text=yt, name="magi", showlegend=True))

y = [fwd_times[k]["magi_rm"] for k in fwd_times if "magi_rm" in fwd_times[k]]
yt = [f"{yy:0.2f}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(fwd_times.keys()), y=y, text=yt, name="magi_rm", showlegend=True))

y = [fwd_times[k]["omni_am"] for k in fwd_times if "omni_am" in fwd_times[k]]
yt = [f"{yy:0.2f}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(fwd_times.keys()), y=y, text=yt, name="omni_am", showlegend=True))

fig.update_layout(title="Forward Time", xaxis_title="Sequence Length", yaxis_title="Time (ms)")
fig.show()

In [8]:
fig = go.Figure()

y = [bwd_times[k]["sdpa"] for k in bwd_times if "sdpa" in bwd_times[k]]
yt = [f"{yy:0.2f}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(bwd_times.keys()), y=y, text=yt, name="sdpa", showlegend=True))

y = [bwd_times[k]["sdpa_c"] for k in bwd_times if "sdpa_c" in bwd_times[k]]
yt = [f"{yy:0.2f}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(bwd_times.keys()), y=y, text=yt, name="sdpa_c", showlegend=True))

y = [bwd_times[k]["magi"] for k in bwd_times if "magi" in bwd_times[k]]
yt = [f"{yy:0.2f}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(bwd_times.keys()), y=y, text=yt, name="magi", showlegend=True))

y = [bwd_times[k]["magi_rm"] for k in bwd_times if "magi_rm" in bwd_times[k]]
yt = [f"{yy:0.2f}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(bwd_times.keys()), y=y, text=yt, name="magi_rm", showlegend=True))

y = [bwd_times[k]["omni_am"] for k in bwd_times if "omni_am" in bwd_times[k]]
yt = [f"{yy:0.2f}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(bwd_times.keys()), y=y, text=yt, name="omni_am", showlegend=True))

fig.update_layout(title="Backward Time", xaxis_title="Sequence Length", yaxis_title="Time (ms)")
fig.show()

In [9]:
fig = go.Figure()

y = [fwd_err[k]["sdpa_c"] for k in fwd_err if "sdpa_c" in fwd_err[k]]
yt = [f"{yy:0.2e}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(fwd_err.keys()), y=y, text=yt, name="sdpa_c", showlegend=True))

y = [fwd_err[k]["magi"] for k in fwd_err if "magi" in fwd_err[k]]
yt = [f"{yy:0.2e}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(fwd_err.keys()), y=y, text=yt, name="magi", showlegend=True))

y = [fwd_err[k]["magi_rm"] for k in fwd_err if "magi_rm" in fwd_err[k]]
yt = [f"{yy:0.2e}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(fwd_err.keys()), y=y, text=yt, name="magi_rm", showlegend=True))

y = [fwd_err[k]["omni_am"] for k in fwd_err if "omni_am" in fwd_err[k]]
yt = [f"{yy:0.2e}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(fwd_err.keys()), y=y, text=yt, name="omni_am", showlegend=True))

fig.update_layout(title="Forward Error", xaxis_title="Sequence Length", yaxis_title="Error")
fig.show()


In [11]:
fig = go.Figure()

y = [bwd_err[k]["sdpa_c"] for k in bwd_err if "sdpa_c" in bwd_err[k]]
yt = [f"{yy:0.2e}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(bwd_err.keys()), y=y, text=yt, name="sdpa_c", showlegend=True))

y = [bwd_err[k]["magi"] for k in bwd_err if "magi" in bwd_err[k]]
yt = [f"{yy:0.2e}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(bwd_err.keys()), y=y, text=yt, name="magi", showlegend=True))

y = [bwd_err[k]["magi_rm"] for k in bwd_err if "magi_rm" in bwd_err[k]]
yt = [f"{yy:0.2e}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(bwd_err.keys()), y=y, text=yt, name="magi_rm", showlegend=True))

y = [bwd_err[k]["omni_am"] for k in bwd_err if "omni_am" in bwd_err[k]]
yt = [f"{yy:0.2e}" for yy in y if yy]
fig.add_trace(go.Bar(x=list(bwd_err.keys()), y=y, text=yt, name="omni_am", showlegend=True))

fig.update_layout(title="Backward Error", xaxis_title="Sequence Length", yaxis_title="Error")
fig.show()

In [12]:
bwd_err

{'32000': {'sdpa': None,
  'sdpa_c': 3.971089932987238e-15,
  'magi': 6.387871514298288e-12,
  'magi_rm': 6.387868424322096e-12,
  'omni_am': 1.6722930277959097e-11},
 '65535': {'sdpa': None,
  'sdpa_c': 1.4166847330185643e-15,
  'magi': 3.048707946794356e-12,
  'magi_rm': 3.0495197702760587e-12,
  'omni_am': 7.163450876316452e-12},
 '256000': {'sdpa': None,
  'sdpa_c': None,
  'magi': None,
  'magi_rm': None,
  'omni_am': None},
 '1024000': {}}